In [2]:
# ========================
# PARTE 0: INSTALAÇÕES
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium > /dev/null
!pip install pandas > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time

# Setup Selenium
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: COLETAR LINKS DE PRODUTOS
# ========================
url = "https://www.leroymerlin.com.br/search?term=Revestimentos%20de%20Parede"
driver.get(url)

todos_links = set()
pagina = 1
LIMITE_PRODUTOS = 10

while len(todos_links) < LIMITE_PRODUTOS:
    print(f"🔄 Página {pagina}...")

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/revestimento']")))
    except TimeoutException:
        print("⚠️ Timeout esperando os produtos.")
        break

    produtos = driver.find_elements(By.CSS_SELECTOR, "a[href*='/revestimento']")
    for p in produtos:
        link = p.get_attribute("href")
        if link:
            todos_links.add(link)
            if len(todos_links) >= LIMITE_PRODUTOS:
                break

    if len(todos_links) >= LIMITE_PRODUTOS:
        break

    try:
        botao_proximo = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Próxima página']")
        if not botao_proximo.is_enabled():
            print("🚫 Botão 'Próxima' desabilitado. Fim da navegação.")
            break
        driver.execute_script("arguments[0].click();", botao_proximo)
        pagina += 1
        time.sleep(2)
    except NoSuchElementException:
        print("❌ Botão 'Próxima página' não encontrado.")
        break

print(f"\n✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRATOR DE DADOS DE PRODUTO
# ========================
def extrair_dados_produto(url):
    driver.get(url)

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        print(f"⚠️ Timeout esperando carregar: {url}")
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}

    try:
        dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text
    except:
        dados['Nome'] = None

    try:
        dados['Código'] = url.split("_")[-1]
    except:
        dados['Código'] = None

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-cy="product-price-integer"]')))
        preco_inteiro = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-integer"]').text
        preco_decimal = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-decimal"]').text
        dados['Preço Promocional'] = f"{preco_inteiro},{preco_decimal}"
    except:
        dados['Preço Promocional'] = None

    try:
        dados['Preço Original'] = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-from-price"]').text
    except:
        dados['Preço Original'] = None

    try:
        embalagem = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-packaging"] strong').text
        unidade = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-packaging"] span:nth-child(2)').text
        dados['Preço por m²'] = f"{embalagem} | {unidade}"
    except:
        dados['Preço por m²'] = None

    try:
        dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content")
    except:
        dados['Descrição'] = None

    try:
        dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content")
    except:
        dados['Imagem'] = None

    try:
        especificacoes = {}
        linhas = driver.find_elements(By.CSS_SELECTOR, 'table tr')
        for linha in linhas:
            try:
                chave = linha.find_element(By.CSS_SELECTOR, 'th').text.strip()
                valor = linha.find_element(By.CSS_SELECTOR, 'td').text.strip()
                especificacoes[chave] = valor
            except:
                continue
        dados['Especificações'] = "\n".join([f"{k}: {v}" for k, v in especificacoes.items()])
    except:
        dados['Especificações'] = None

    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETA DOS DADOS DE TODOS OS PRODUTOS
# ========================
todos_dados = []

for idx, link in enumerate(sorted(todos_links), 1):
    print(f"🔍 Coletando {idx}/{len(todos_links)}: {link}")
    try:
        dados = extrair_dados_produto(link)
        todos_dados.append(dados)
    except Exception as e:
        print(f"⚠️ Erro ao coletar {link}: {e}")

# ========================
# PARTE 5: SALVAR CSV
# ========================
df = pd.DataFrame(todos_dados, columns=[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"])

df.to_csv("produtos_leroy.csv", index=False, encoding='utf-8-sig', sep='\t')

print("\n✅ Arquivo CSV gerado: produtos_leroy.csv")
driver.quit()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔄 Página 1...

✅ Total de links coletados: 10

🔍 Coletando 1/10: https://www.leroymerlin.com.br/revestimento-acetinado-32x55cm-32700-cristofoletti_92358322
🔍 Coletando 2/10: https://www.leroymerlin.com.br/revestimento-brilhante-33x60cm-campestre-embramaco_92283905
🔍 Coletando 3/10: https://www.leroymerlin.com.br/revestimento-esmaltado-oasis-perola-borda-arredondada-bege-31x54cm-savane_92333843
🔍 Coletando 4/10: https://www.leroymerlin.com.br/revestimento-interno-acetinado-borda-arredondada-31x54cm-prisma-roble-savane_91081466
🔍 Coletando 5/10: https://www.leroymerlin.com.br/revestimento-interno-amadeirado-acetinado-borda-reta-32x60cm-rig-legno-biancogres_90543691
🔍 Coletando 6/10: https://www.leroymerlin.com.